# 8. Replacement Level, VORP, and Draft Tiers

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `07_projections.ipynb` first.

A projection (`07_projections.ipynb`) ranks players against each other. It says nothing about *where the bench begins* -- the point past which two players are interchangeable because either is a free-agent-caliber replacement anyway. `nuclearff.valuation` answers that with value-based drafting: replacement level, VORP, VONA, and discrete draft tiers. Mirrors the [Valuation](https://nolmacdonald.github.io/nuclearff/tutorial/08_valuation.html) docs page.

## Scoring and projecting a real board

Everything below needs a projected-points table first. Reuse the `ScoringEngine` from `05_scoring_engine.ipynb` and two wiring functions `09_auction_draft_board.ipynb` covers in full -- `score_seasons` (this league's own rules over real seasonal stats) and `project_points` (recency-weighted realized points, the same idea as `07_projections.ipynb`'s `blend_projection` but operating on whole fantasy-point totals instead of one rate column):

In [1]:
import polars as pl

from nuclearff.config import load_league_config
from nuclearff.nflverse import load_seasonal_skill_stats
from nuclearff.pipeline import project_points, score_seasons
from nuclearff.scoring import ScoringEngine

league_cfg = load_league_config("./demo/configs/leagues/1367225133634191360.yaml")
engine = ScoringEngine(league_cfg.scoring)

stats = load_seasonal_skill_stats([2023, 2024, 2025])
scored = score_seasons(stats, engine)
projections = project_points(scored, 2026)

ScoringEngine has no mapping for 48 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, blk_kick_ret_yd=0.04, def_3_and_out=0.25, def_4_and_stop=0.5, def_pr_yd=0.04, def_st_ff=1.0, def_st_fum_rec=1.0, def_st_td=6.0, def_td=6.0, ff=1.0, fg_ret_yd=0.04, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=4.0, fgm_50p=5.0, fgmiss=-1.0, fgmiss_0_19=-1.0, fgmiss_20_29=-1.0, fum_rec=2.0, fum_rec_td=6.0, fum_ret_yd=0.04, int=2.0, int_ret_yd=0.04, kr_yd=0.04, pr_yd=0.04, pts_allow_0=5.0, pts_allow_14_20=1.0, pts_allow_1_6=4.0, pts_allow_28_34=-1.0, pts_allow_35p=-4.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_ff=1.0, st_fum_rec=1.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0, yds_allow_0_100=5.0, yds_allow_100_199=3.0, yds_allow_200_299=2.0, yds_allow_350_399=-1.0, yds_allow_400_449=-3.0, yds_allow_450_499=-5.0, yds_allow_500_549=-6.0, yds_allow_550p=-7.0


`projections` now has one row per player with a `value_estimate` column (the recency-weighted points estimate) plus identity columns.

## Replacement level

`replacement_points` returns the projected points of the replacement-level player at a position, under one of two named baselines:

In [2]:
from nuclearff.valuation import replacement_points

vols = replacement_points(
    projections,
    league_cfg,
    position="WR",
    baseline="vols",
    proj_column="value_estimate",
)
vorp_baseline = replacement_points(
    projections,
    league_cfg,
    position="WR",
    baseline="vorp",
    proj_column="value_estimate",
)
print(f"WR replacement level (vols): {vols:.1f}")
print(f"WR replacement level (vorp): {vorp_baseline:.1f}")

WR replacement level (vols): 168.0
WR replacement level (vorp): 135.8


Recall from `02_configuration.ipynb`: **VOLS** (Value Over Last Starter) uses the league's actual starter demand as the replacement rank; **VORP** (Value Over Replacement Player) goes deeper into the bench on the assumption that real waiver-wire depth, not the last nominal starter, is where a player actually becomes replaceable. Neither is more "correct" -- they're two different, named assumptions, and VORP's bench-stash fraction is explicitly an uncalibrated heuristic (see `LeagueConfig.replacement_rank`'s own docstring).

## VORP

`vorp` adds a `vorp` column: each player's value estimate minus the replacement level:

In [3]:
from nuclearff.valuation import vorp

valued = vorp(
    projections,
    league_cfg,
    position="WR",
    baseline="vols",
    proj_column="value_estimate",
)
top_wr = (
    valued.filter(pl.col("position") == "WR")
    .sort("vorp", descending=True)
    .select(["player_display_name", "value_estimate", "vorp"])
)
top_wr.head(8)

player_display_name,value_estimate,vorp
str,f64,f64
"""Ja'Marr Chase""",330.244,162.284
"""Amon-Ra St. Brown""",323.634,155.674
"""Puka Nacua""",309.18,141.22
"""Jaxon Smith-Njigba""",285.81,117.85
"""CeeDee Lamb""",260.11,92.15
"""George Pickens""",237.03,69.07
"""Davante Adams""",236.92,68.96
"""Justin Jefferson""",236.434,68.474


Watch for the real gap in the output: the top handful of players typically separate from the next group by a wide VORP margin, while the next several bunch up close together -- exactly the kind of structure draft tiers (below) are meant to surface explicitly instead of leaving it implicit in a sorted list.

## VONA: value over next available

Replacement level answers "how much better is this player than a free-agent floor." `vona` answers a sharper, in-the-moment question: "how much do I lose if I wait and take this position at my *next* pick instead?" It needs the set of players already drafted and the number of picks until your next turn:

In [4]:
from nuclearff.valuation import vona

wr_pool = valued.filter(pl.col("position") == "WR")
drafted_ids = set(wr_pool.sort("vorp", descending=True).head(3)["player_id"])

with_vona = vona(
    wr_pool, drafted_ids, next_pick_gap=8, position="WR", proj_column="value_estimate"
)
with_vona.sort("vona", descending=True).select(
    ["player_display_name", "value_estimate", "vona"]
).head(6)

player_display_name,value_estimate,vona
str,f64,f64
"""Puka Nacua""",309.18,null
"""Amon-Ra St. Brown""",323.634,null
"""Ja'Marr Chase""",330.244,null
"""Jaxon Smith-Njigba""",285.81,60.03
"""CeeDee Lamb""",260.11,34.33
"""George Pickens""",237.03,11.25


The three already-`drafted_ids` correctly get `vona = null` -- they're off the board, not a real choice anymore. A high VONA for the next-best player says: if you pass on them now, the best WR still around at your next turn (8 picks later) will cost you that many points of value -- a much sharper signal than VORP alone for deciding whether to reach.

## Draft tiers

A sorted VORP list still asks you to eyeball where one tier of value ends and the next begins. `assign_tiers` groups a continuous value column into discrete tiers via k-means, auto-selecting `k` by silhouette score unless you fix it:

In [5]:
from nuclearff.valuation import assign_tiers

ranked = valued.filter(pl.col("position") == "WR").sort("vorp", descending=True).head(8)
tiers = assign_tiers(ranked["vorp"].to_list())
list(zip(ranked["player_display_name"], tiers, strict=True))

[("Ja'Marr Chase", 1),
 ('Amon-Ra St. Brown', 1),
 ('Puka Nacua', 1),
 ('Jaxon Smith-Njigba', 1),
 ('CeeDee Lamb', 2),
 ('George Pickens', 2),
 ('Davante Adams', 2),
 ('Justin Jefferson', 2)]

The real top tier separates cleanly from the rest -- a tier boundary computed from the actual data's clustering, not a fixed "top 5 / next 5" rule that would split two closely-valued players arbitrarily.

> **Why this matters:** it's worth actually testing `assign_tiers` on a deliberately shuffled input once, the way this project's own test suite does -- k-means assigns arbitrary numeric labels to clusters, so "tier 1" isn't guaranteed to be the *highest*-value cluster unless something remaps it afterward. `assign_tiers` does that remapping for you, but it's exactly the kind of thing that's easy to get backwards if you ever reimplement tiering yourself.

## What's Next

`09_auction_draft_board.ipynb` puts all of this together end to end -- including converting VORP into real auction dollars -- against a league that actually runs an auction draft, plus renders the full report as CSV, markdown, and styled PNG tables.

## See Also

- `07_projections.ipynb` -- the projection this chapter turns into a draft value.
- [Valuation](https://nolmacdonald.github.io/nuclearff/tutorial/08_valuation.html) -- the matching docs page.